# Stage A — FP4 vs FP8 GEMM crossover at M=128 (the MLA decode shape)

**Goal:** Does native FP4 tensor-core compute beat FP8 at the MLA decode QK-GEMM shape
(M=128, K in {256, 512, 576}, N = context length 1K..524K)?

**Prediction (pre-registered): NO.** Decode is HBM/work-starved, not compute-bound. At M=128 the
QK GEMM has arithmetic intensity ~135-176 FLOP/byte across the whole N sweep (the F32 output C[M,N]
dominates bytes and scales with N just like operand B), which sits far below both roofline ridges
(FP8 ridge 625, FP4 ridge 1875). So the bottleneck is memory/pipeline-fill and FP4's 3x compute
peak cannot convert. This mirrors the whole-kernel v6->v12 finding (per-CTA / work-starvation).

**Kill condition:** FP4 <= FP8 TFLOP/s at ALL shapes -> publish the negative in paper Section 5.3
("the M=128 packing advantage is a red herring for decode; native FP4 compute does not help").

**Proceed condition:** FP4 > FP8 at some shape -> record the crossover, go to Stage B (accuracy).

**Only QK is testable.** PV cannot be FP4 (softmax collapses P; proven in v10), so Stage A
benchmarks the QK GEMM shape only.

**Hardware:** B300 SXM (sm_103a), CUTLASS v4.5.2, CUDA 12.9+. This is the *run-of-record* notebook:
Run-All on the box, then commit the executed copy + `stage_a_results.csv`.

---
**Measurement-consistency contract (the scientific crux).** The verdict is a TFLOP/s *ratio*
FP4/FP8, so both precisions MUST be measured with matched output dtype through the same harness.
CUTLASS example 72b writes an FP4 output; naively comparing that against an F32-output FP8 run would
bias the ratio. Primary path runs BOTH through `cutlass_profiler` with identical `--C/--D`. Fallback
keeps dtypes matched (bf16 <-> bf16). Every result row records `source`, `out_dtype`, `kernel_name`
so the comparison is auditable. Because FP4 and FP8 are measured back-to-back on the same box, a
clock lock is nice-to-have but not gating (DVFS cancels in the ratio).

## 0. Environment verify (fail loud before burning build time)

In [ ]:
import os, sys, subprocess, re

# Make the repo importable (roofline.archs) regardless of nbconvert cwd.
_cwd = os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(_cwd, '..')) if os.path.basename(_cwd) == 'notebooks' else _cwd
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('repo root:', REPO_ROOT)

# GPU identity
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU     :', smi.stdout.strip())

# CUDA toolkit version
nvcc = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
m = re.search(r'release (\d+)\.(\d+)', nvcc)
cuda_major, cuda_minor = (int(m.group(1)), int(m.group(2))) if m else (0, 0)
print(f'nvcc    : {cuda_major}.{cuda_minor}')

# Torch device capability (must be (10, 3) for B300/sm_103)
try:
    import torch
    cap = torch.cuda.get_device_capability()
    print('torch cap:', cap)
except Exception as e:
    cap = None
    print('torch not available for cap check:', e)

# --- Hard gates: STOP now rather than waste a CUTLASS build ---
assert 'B300' in smi.stdout, f'Expected a B300; got: {smi.stdout.strip()!r}'
assert (cuda_major, cuda_minor) >= (12, 9), f'Need CUDA >= 12.9 for sm_103; got {cuda_major}.{cuda_minor}'
if cap is not None:
    assert cap == (10, 3), f'Expected compute capability (10, 3) for sm_103; got {cap}'
print('\nEnvironment OK: B300 / CUDA >= 12.9 / sm_103.')

## 1. Build CUTLASS v4.5.2 (profiler + example 72a/72b)

Builds the three targets needed by both the primary (profiler) and fallback (example 72) paths.
Cache-guarded: a completed build is reused. `-DCUTLASS_ENABLE_EXAMPLES=ON` is required for 72a/72b,
`-DCUTLASS_ENABLE_PROFILER=ON` for the primary path. Only these three targets are built (not all
examples) to keep the build within ~40 GB disk / avoid OOM.

In [ ]:
CUTLASS_DIR = os.path.expanduser('~/cutlass')
BUILD_DIR   = os.path.join(CUTLASS_DIR, 'build')
EX_DIR      = os.path.join(BUILD_DIR, 'examples', '72_blackwell_narrow_precision_gemm')

PROFILER = os.path.join(BUILD_DIR, 'tools', 'profiler', 'cutlass_profiler')
EX72A    = os.path.join(EX_DIR, '72a_blackwell_nvfp4_bf16_gemm')   # NVFP4 -> BF16 (fallback FP4 path)
EX72B    = os.path.join(EX_DIR, '72b_blackwell_nvfp4_nvfp4_gemm')  # NVFP4 -> NVFP4

def _sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = '\n'.join((r.stdout + r.stderr).splitlines()[-8:])
    print(tail)
    return r.returncode

CUTLASS_TAG = 'v4.5.2'   # latest release as of 2026-07; NOTE: there is no v4.6.0 tag.

def _valid_cutlass(d):
    return os.path.exists(os.path.join(d, 'CMakeLists.txt'))

if not _valid_cutlass(CUTLASS_DIR):
    _sh(f'rm -rf {CUTLASS_DIR}')   # wipe any stale/empty dir from a failed clone
    rc = _sh(f'git clone --depth 1 --branch {CUTLASS_TAG} '
             f'https://github.com/NVIDIA/cutlass.git {CUTLASS_DIR}')
    if not _valid_cutlass(CUTLASS_DIR):
        print(f'tag {CUTLASS_TAG} clone failed; falling back to default branch (latest)')
        _sh(f'rm -rf {CUTLASS_DIR}')
        _sh(f'git clone --depth 1 https://github.com/NVIDIA/cutlass.git {CUTLASS_DIR}')
    assert _valid_cutlass(CUTLASS_DIR), 'CUTLASS clone failed (no CMakeLists.txt)'

os.makedirs(BUILD_DIR, exist_ok=True)

if not os.path.exists(os.path.join(BUILD_DIR, 'CMakeCache.txt')):
    rc = _sh(f'cd {BUILD_DIR} && cmake .. '
             f'-DCUTLASS_NVCC_ARCHS="103a" '
             f'-DCUTLASS_ENABLE_PROFILER=ON '
             f'-DCUTLASS_ENABLE_EXAMPLES=ON '
             f'-DCUTLASS_ENABLE_TESTS=OFF '
             f'-DCMAKE_BUILD_TYPE=Release')
    assert rc == 0, 'cmake configure failed (see tail above)'

# Build only the targets we need.
for tgt, path in [('cutlass_profiler', PROFILER),
                  ('72a_blackwell_nvfp4_bf16_gemm', EX72A),
                  ('72b_blackwell_nvfp4_nvfp4_gemm', EX72B)]:
    if os.path.exists(path):
        print(f'cached: {tgt}')
        continue
    rc = _sh(f'cd {BUILD_DIR} && make {tgt} -j$(nproc)')
    # Non-fatal per-target: the sweep guards on which binaries actually exist.
    print(f'{tgt}: {"OK" if os.path.exists(path) else "MISSING (rc=%d)" % rc}')

print()
for name, path in [('profiler', PROFILER), ('72a', EX72A), ('72b', EX72B)]:
    print(f'{name:9s}: {"OK" if os.path.exists(path) else "MISSING"}  {path}')
assert os.path.exists(PROFILER) or os.path.exists(EX72A), \
    'Need at least the profiler OR example 72a to run Stage A.'


## 2. Sweep definition

In [ ]:
# QK GEMM shapes for MLA decode. M=128 = all h_q query heads packed (the MLA advantage).
M_VALUES = [128]
K_VALUES = [256, 512, 576]                                  # head-dim variants (latent-only .. absorbed)
N_VALUES = [1024, 8192, 32768, 131072, 524288]             # context lengths N_k
WARMUP   = 5
ITERS    = 20

# All K are multiples of 16 (block-scale vector) and 64 (MMA-atom K): 256=16x16, 512=32x16, 576=36x16.
# All N are multiples of 16. So no alignment padding is expected; per-shape failures are still guarded.
N_SHAPES = len(M_VALUES) * len(K_VALUES) * len(N_VALUES)
print(f'Sweep: {len(M_VALUES)} M x {len(K_VALUES)} K x {len(N_VALUES)} N = {N_SHAPES} shapes x 2 precisions'
      f' = {N_SHAPES * 2} runs')

## 3. Run the FP4-vs-FP8 sweep (matched output dtype)

**Primary path:** both precisions via `cutlass_profiler` with identical `--C=f32 --D=f32`.
**Guard:** if the profiler does not enumerate a block-scaled NVFP4 kernel for a shape, fall back to
example 72a (NVFP4 -> BF16) for FP4 and profiler `e4m3 -> bf16` for FP8 — still matched output dtype.
Each row records `source` + `out_dtype` + `kernel_name` for auditability. Every shape is wrapped in
try/except so one failure never aborts the sweep.

In [ ]:
import glob, math, csv, tempfile

def _profiler_gflops(A, B, Ct, Dt, M, N, K, timeout=180):
    """Run cutlass_profiler for one shape/precision. Return (best_gflops, kernel_name) or (None, reason)."""
    if not os.path.exists(PROFILER):
        return None, 'no-profiler-binary'
    with tempfile.TemporaryDirectory() as td:
        out = os.path.join(td, 'res')
        cmd = [PROFILER, '--operation=gemm',
               f'--m={M}', f'--n={N}', f'--k={K}',
               f'--A={A}', f'--B={B}', f'--C={Ct}', f'--D={Dt}',
               f'--warmup-iterations={WARMUP}', f'--profiling-iterations={ITERS}',
               '--providers=cutlass', f'--output={out}']
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            return None, 'timeout'
        # Profiler writes <out>.gemm.csv (name may vary); pick the freshest csv it produced.
        csvs = sorted(glob.glob(out + '*.csv'), key=os.path.getmtime)
        if not csvs:
            # No kernels enumerated for this precision (e.g. block-scaled nvfp4 not in profiler).
            reason = 'no-csv'
            if r.returncode != 0:
                reason = 'rc=%d:%s' % (r.returncode, (r.stderr or r.stdout)[-120:].replace('\n', ' '))
            return None, reason
        best_g, best_k = 0.0, ''
        with open(csvs[-1]) as f:
            rd = csv.DictReader(f)
            # Find the GFLOPs column and an operation/kernel-name column case-insensitively.
            gcol = kcol = None
            for col in (rd.fieldnames or []):
                lc = col.strip().lower()
                if gcol is None and lc in ('gflops', 'gflop/s', 'gflops/s'):
                    gcol = col
                if kcol is None and lc in ('operation', 'kernel', 'kernel_name', 'name'):
                    kcol = col
            if gcol is None:
                return None, 'no-gflops-col:' + ','.join(rd.fieldnames or [])
            for row in rd:
                try:
                    g = float(row[gcol])
                except (TypeError, ValueError):
                    continue
                if g > best_g:
                    best_g, best_k = g, (row.get(kcol, '') if kcol else '')
        if best_g <= 0:
            return None, 'no-positive-gflops'
        return best_g, (best_k or 'cutlass_gemm')


def _example72a_gflops(M, N, K, timeout=180):
    """FP4 fallback: NVFP4 -> BF16 GEMM via example 72a. Return (gflops, kernel_name) or (None, reason)."""
    if not os.path.exists(EX72A):
        return None, 'no-72a-binary'
    cmd = [EX72A, f'--m={M}', f'--n={N}', f'--k={K}', f'--iterations={ITERS}']
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, 'timeout'
    txt = r.stdout + r.stderr
    # Examples print e.g. "GFLOPs: 1234.5" and/or "Avg runtime: 0.123 ms". Prefer GFLOPs, else derive.
    mg = re.search(r'GFLOP[s/]*\s*[:=]\s*([0-9.]+)', txt, re.I)
    if mg:
        return float(mg.group(1)), '72a_nvfp4_bf16'
    mt = re.search(r'([0-9.]+)\s*ms', txt)
    if mt and float(mt.group(1)) > 0:
        gflops = (2.0 * M * N * K) / (float(mt.group(1)) * 1e-3) / 1e9
        return gflops, '72a_nvfp4_bf16'
    return None, ('parse-fail:' + txt[-120:].replace('\n', ' ')) if r.returncode == 0 \
        else 'rc=%d:%s' % (r.returncode, txt[-120:].replace('\n', ' '))

In [ ]:
results = []   # one row per (shape, precision)

def _add(prec, M, K, N, gflops, source, out_dtype, kernel, error=''):
    results.append({'precision': prec, 'M': M, 'K': K, 'N': N,
                    'tflops': (gflops / 1e3) if gflops else float('nan'),
                    'source': source, 'out_dtype': out_dtype,
                    'kernel_name': kernel, 'error': error})

for M in M_VALUES:
    for K in K_VALUES:
        for N in N_VALUES:
            tag = f'M={M} K={K} N={N}'
            # ---- FP4 first: decide primary-vs-fallback from whether the profiler has a block-scaled kernel.
            fp4_primary_ok = False
            try:
                g4, k4 = _profiler_gflops('e2m1', 'e2m1', 'f32', 'f32', M, N, K)
            except Exception as e:
                g4, k4 = None, f'exc:{e}'
            if g4 is not None and ('nvfp4' in k4.lower() or 'e2m1' in k4.lower() or 'f4' in k4.lower()):
                fp4_primary_ok = True
                _add('fp4_e2m1', M, K, N, g4, 'profiler', 'f32', k4)
                print(f'[FP4 profiler ] {tag}: {g4/1e3:.1f} TFLOP/s  ({k4[:40]})')
            else:
                # Fallback: example 72a (NVFP4 -> BF16). Output dtype becomes bf16.
                try:
                    g4b, k4b = _example72a_gflops(M, N, K)
                except Exception as e:
                    g4b, k4b = None, f'exc:{e}'
                if g4b is not None:
                    _add('fp4_e2m1', M, K, N, g4b, 'example72a', 'bf16', k4b)
                    print(f'[FP4 ex72a    ] {tag}: {g4b/1e3:.1f} TFLOP/s  (profiler miss: {k4})')
                else:
                    _add('fp4_e2m1', M, K, N, None, 'none', '-', '', error=f'profiler:{k4} | 72a:{k4b}')
                    print(f'[FP4 FAIL     ] {tag}: {k4} | {k4b}')

            # ---- FP8: match the FP4 output dtype (f32 if FP4 used the profiler, else bf16).
            fp8_out = 'f32' if fp4_primary_ok else 'bf16'
            try:
                g8, k8 = _profiler_gflops('e4m3', 'e4m3', fp8_out, fp8_out, M, N, K)
            except Exception as e:
                g8, k8 = None, f'exc:{e}'
            if g8 is not None:
                _add('fp8_e4m3', M, K, N, g8, 'profiler', fp8_out, k8)
                print(f'[FP8 profiler ] {tag}: {g8/1e3:.1f} TFLOP/s  (out={fp8_out}, {k8[:32]})')
            else:
                _add('fp8_e4m3', M, K, N, None, 'none', '-', '', error=f'profiler:{k8}')
                print(f'[FP8 FAIL     ] {tag}: {k8}')

print(f'\nDone: {len(results)} rows')

## 3b. Fallback escape hatch (documented, normally unused)

The primary profiler path + example 72a/72b cover Stage A. Only if BOTH the profiler AND example 72a
fail to produce an FP4 number for a shape do you need a bespoke kernel. In that case, build and adapt
`~/cutlass/examples/72_blackwell_narrow_precision_gemm/72b_blackwell_nvfp4_nvfp4_gemm.cu` directly
(it accepts `--m/--n/--k`); pair it with a BF16-output FP8 baseline to keep output dtype matched. Do
NOT compare 72b's FP4 output against an F32-output FP8 run — that biases the TFLOP/s ratio. This cell
is intentionally left as a note, not code.

## 4. Analyze + verdict

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df_valid = df[df['tflops'].notna()].copy()
print(f'valid rows: {len(df_valid)} / {len(df)}')

if len(df_valid) == 0:
    print('No valid results -- inspect the sweep log above.')
else:
    piv = df_valid.pivot_table(index=['M', 'K', 'N'], columns='precision', values='tflops').reset_index()
    have_both = ('fp4_e2m1' in piv.columns) and ('fp8_e4m3' in piv.columns)
    if have_both:
        piv['fp4_over_fp8'] = piv['fp4_e2m1'] / piv['fp8_e4m3']
        piv['winner'] = piv['fp4_over_fp8'].apply(
            lambda r: 'FP4' if r > 1.05 else ('FP8' if r < 0.95 else 'TIE'))
    print('\n=== FP4 vs FP8 TFLOP/s at M=128 ===')
    print(piv.to_string(index=False))

    print('\n=== VERDICT ===')
    if have_both and piv['fp4_over_fp8'].notna().any():
        max_ratio = piv['fp4_over_fp8'].max()
        fp4_wins = (piv['winner'] == 'FP4').sum()
        if max_ratio <= 1.05:
            print(f'>>> KILL. FP4 never beats FP8 (max FP4/FP8 = {max_ratio:.3f} <= 1.05).')
            print('>>> Native FP4 compute does not help MLA decode: the QK GEMM at M=128 is')
            print('>>> memory/work-starved, not compute-bound -- as predicted. Negative -> paper Section 5.3.')
        else:
            crossover = piv.loc[piv['fp4_over_fp8'] > 1.05, ['M', 'K', 'N', 'fp4_over_fp8']]
            print(f'>>> PROCEED. FP4 beats FP8 at {fp4_wins} shape(s) (max ratio {max_ratio:.3f}).')
            print(crossover.to_string(index=False))
            print('>>> Record the crossover shape -> Stage B (accuracy on real KV).')
    else:
        print('>>> INCONCLUSIVE: missing one precision. Check `source`/`error` columns in df.')

## 5. Roofline context (predict the null before the numbers confirm it)

In [ ]:
# Pull B300 constants from the repo's roofline model (not hardcoded), with a literal fallback.
try:
    from roofline.archs import B300
    HBM_BW   = B300.hbm_bw_gbps * 1e9   # 8e12 B/s
    FP8_PEAK = B300.fp8_tc_flops        # 5e15
    FP4_PEAK = B300.fp4_tc_flops        # 15e15
    print('roofline constants from roofline.archs.B300')
except Exception as e:
    HBM_BW, FP8_PEAK, FP4_PEAK = 8e12, 5e15, 15e15   # mirrors archs.py B300
    print('roofline.archs import failed (%s); using literal B300 constants' % e)

FP8_RIDGE = FP8_PEAK / HBM_BW   # 625
FP4_RIDGE = FP4_PEAK / HBM_BW   # 1875
print(f'ridges: FP8={FP8_RIDGE:.0f}  FP4={FP4_RIDGE:.0f} FLOP/byte  (HBM {HBM_BW/1e12:.0f} TB/s)\n')

rows = []
for _, r in df_valid.iterrows():
    M, K, N, prec, tflops = r['M'], r['K'], r['N'], r['precision'], r['tflops']
    b = 0.5625 if 'fp4' in prec else 1.0                 # NVFP4 = 0.5 data + 0.0625 micro-scale
    peak, ridge = (FP4_PEAK, FP4_RIDGE) if 'fp4' in prec else (FP8_PEAK, FP8_RIDGE)
    flops = 2 * M * N * K
    hbm_bytes = M * K * b + K * N * b + M * N * 4        # +F32 output C[M,N]
    ai = flops / hbm_bytes
    limiter = 'COMPUTE' if ai > ridge else 'HBM'
    pct_peak = tflops * 1e12 / peak * 100
    rows.append({'precision': prec, 'M': M, 'K': K, 'N': N, 'AI': round(ai, 1),
                 'ridge': int(ridge), 'limiter': limiter,
                 'tflops': round(tflops, 1), 'pct_peak': round(pct_peak, 2)})
roof = pd.DataFrame(rows).sort_values(['precision', 'K', 'N'])
print(roof.to_string(index=False))
print('\nExpected: every shape HBM-bound (AI << ridge) and pct_peak in the low single digits ->')
print('the roofline predicts FP4 cannot convert its 3x compute peak. If any row shows COMPUTE or a')
print('near-peak pct_peak, the measurement needs re-inspection.')

## 6. Save results for the paper

In [ ]:
# Write next to this notebook (notebooks/), full rows incl. failures for provenance.
OUT_CSV = os.path.join(_cwd, 'stage_a_results.csv')
df.to_csv(OUT_CSV, index=False)
print('wrote', OUT_CSV, f'({len(df)} rows)')

# One-line verdict banner for quick eyeballing in the committed notebook.
if len(df_valid) and 'fp4_over_fp8' in globals().get('piv', pd.DataFrame()).columns:
    mr = piv['fp4_over_fp8'].max()
    banner = 'KILL (FP4<=FP8 everywhere)' if mr <= 1.05 else f'PROCEED (FP4 wins, max {mr:.2f}x)'
    print('VERDICT:', banner, f'| max FP4/FP8 = {mr:.3f}')
else:
    print('VERDICT: INCONCLUSIVE (see df.error)')
print('\nNext: paste into results.md / decisions.md Stage A; commit the executed notebook + CSV.')